In [ ]:
!uv pip install "langchain-azure-ai[tools,opentelemetry]" azure-identity python-dotenv

In [ ]:
import os
import sys
 
from dotenv import load_dotenv
 
# Load .env BEFORE importing/instantiating anything that reads os.environ
load_dotenv()
 
from azure.core.exceptions import ClientAuthenticationError
from azure.identity import DefaultAzureCredential
from langchain_core.messages import HumanMessage
 
from langchain_azure_ai.agents import AgentServiceFactory

In [2]:
def require_env(name: str) -> str:
    """Fetch a required env var or exit with a clear error."""
    value = os.environ.get(name)
    if not value:
        sys.exit(f"Missing required environment variable: {name} (check your .env file)")
    return value

In [3]:
project_endpoint = require_env("AZURE_AI_PROJECT_ENDPOINT")
agent_name = require_env("AGENT_NAME")
agent_version = os.environ.get("AGENT_VERSION", "latest")

credential = DefaultAzureCredential()

# Fail fast with a clear message if auth isn't set up correctly,
# rather than letting it surface as a confusing "agent not found" error.
try:
    credential.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as exc:
    sys.exit(
        "Azure authentication failed. Fix one of the following and retry:\n"
        "  - Run `az login` (Azure CLI auth), or\n"
        "  - Set AZURE_CLIENT_ID / AZURE_TENANT_ID / AZURE_CLIENT_SECRET in .env "
        "(service principal auth)\n\n"
        f"Original error: {exc}"
    )


In [ ]:
factory = AgentServiceFactory(
    project_endpoint=project_endpoint,
    credential=credential,
)

try:
    agent_node = factory.get_agent_node(
        name=agent_name,
        version=agent_version,
    )
except ValueError as exc:
    sys.exit(
        f"Could not find agent '{agent_name}' (version={agent_version!r}) in the "
        f"connected project. Double-check the name/version in the Foundry portal.\n\n"
        f"Original error: {exc}"
    )

response = agent_node.invoke(
    {"messages": [HumanMessage(content="What are the preferred antihypertensive agents for chronic hypertension in pregnant or planning-to-become-pregnant patients?")]}
)
print(response)

conversation_id = response.get("azure_ai_agents_conversation_id")
if conversation_id:
    print("\nazure_ai_agents_conversation_id:", conversation_id)